# 51 — Phase 1 Bundle A: BGE-M3 retriever swap

**Goal**: see if swapping the metadata-dense sub of the wRRF retriever from Qwen3-Embedding-0.6B to **BGE-M3** moves the **official-evaluator nDCG@20**.

**The single-axis change** (vs config 110 baseline):
- `retrieval_type: wrrf_bm25_dense_lyrics_v1` → `wrrf_bm25_dense_lyrics_bge_m3_v1`

Everything else is identical: same BM25 (5-field), same lyrics-dense (Qwen3-0.6B), same CMQR rewriter, same ProRank reranker, same Qwen-1.5B responder. Any Δ in nDCG@20 is attributable to the metadata-dense embedder swap.

**Why offline-first** (per `feedback_offline_eval_must_match_online.md`): we run the EXACT same pipeline Blind-A uses (`run_inference_devset.py`) and score with the EXACT same evaluator (`music-crs-evaluator/evaluate_devset.py`). The nDCG@20 number this notebook produces is the same metric the leaderboard uses — just on dev (8000 turns) instead of Blind-A (80 turns), so the signal is much tighter.

**Decision rule**: if Δ nDCG@20 ≥ +0.02 with the Bundle A config, ship it as the next Blind-A submission. If smaller, keep BGE-M3 in the ensemble pool but don't burn a solo Blind-A slot.

**Wallclock estimate (A100-40GB / Blackwell-95GB)**:
| Step | Time | One-time? |
|---|---|---|
| Embed 47K-track catalog with BGE-M3 | ~30–60 min | yes (cached on Drive) |
| Run baseline (config 110) inference | ~30–45 min | yes (cached) |
| Run experiment (config 120) inference | ~30–45 min | re-run for each new experiment |
| Evaluate both (official evaluator) | ~5 min | re-run as needed |

First-time end-to-end ~1.5–2.5 hr. Subsequent experiment iterations ~30–45 min.

In [ ]:
# 1) GPU check.
!nvidia-smi | head -10

In [ ]:
# 2) Clone fresh-model branch.
BRANCH = 'fresh-model'
!rm -rf /content/recsys2026
!git clone -b {BRANCH} https://github.com/orrimoch/recsys2026-lora-tutorial.git /content/recsys2026
%cd /content/recsys2026

In [ ]:
# 3) HF auth + Drive mount (so the BGE-M3 catalog cache survives across runtimes).
import os
from google.colab import userdata
try:
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    print('HF_TOKEN set from Colab secrets.')
except Exception as e:
    print('NO HF_TOKEN — set it in Colab secrets before running cell 5.', e)

from google.colab import drive
drive.mount('/content/drive')
os.environ['HF_HOME'] = '/content/drive/MyDrive/hf_cache'
print('HF_HOME =', os.environ['HF_HOME'])

# Persist the BGE-M3 catalog embedding cache on Drive so we don't re-embed on every runtime.
DRIVE_EMBED_DIR = '/content/drive/MyDrive/recsys2026_embed_cache'
os.makedirs(DRIVE_EMBED_DIR, exist_ok=True)
# Symlink Drive cache into the repo cache_dir so the retriever finds it.
REPO_CACHE_DIR = '/content/recsys2026/music-crs-baselines/experiments/cache/dense_local'
os.makedirs(os.path.dirname(REPO_CACHE_DIR), exist_ok=True)
if os.path.islink(REPO_CACHE_DIR) or os.path.exists(REPO_CACHE_DIR):
    !rm -rf {REPO_CACHE_DIR}
!ln -s {DRIVE_EMBED_DIR} {REPO_CACHE_DIR}
print('symlinked', REPO_CACHE_DIR, '->', DRIVE_EMBED_DIR)

In [ ]:
# 4) Install deps (note: sentence-transformers added vs Phase 0 deps).
!pip install -q --upgrade transformers datasets 'pandas<3.0' tqdm omegaconf pyyaml bm25s scipy numpy sentence-transformers
# vLLM for the responder (used by full inference). If install is slow, this can be skipped
# and we set use_vllm: false in the configs.
!pip install -q --upgrade 'vllm>=0.6.0' || echo 'vLLM install failed; will fall back to use_vllm: false'

In [ ]:
# 5) Embed the 47K-track catalog with BGE-M3 (one-time, ~30-60 min).
# Skips automatically if the cache exists on Drive.
import os
EMBED_CACHE = f"{REPO_CACHE_DIR}/BAAI_bge-m3/bge-m3-metadata/track_embeddings.pkl"
if os.path.exists(EMBED_CACHE):
    print(f'Cache already exists at {EMBED_CACHE} — skipping embed step.')
else:
    print('No cache found — embedding catalog (this takes ~30-60 min on A100/Blackwell).')
    !python scripts/embed_catalog.py \
        --model BAAI/bge-m3 \
        --label bge-m3-metadata \
        --batch-size 64
    print('Embed complete.')

In [ ]:
# 6) Run BASELINE (config 110) full pipeline inference, if not already cached.
# This is the same code path Blind-A submissions use.
%cd /content/recsys2026/music-crs-baselines
BASELINE_TID = '110-prorank-rerank-devset'
BASELINE_PRED = f'exp/inference/devset/{BASELINE_TID}.json'
import os
if os.path.exists(BASELINE_PRED):
    print(f'Baseline predictions already exist at {BASELINE_PRED} — skipping.')
else:
    print('Running baseline inference (~30-45 min)...')
    !python run_inference_devset.py --tid {BASELINE_TID}
%cd /content/recsys2026

In [ ]:
# 7) Run EXPERIMENT (config 120) full pipeline inference.
%cd /content/recsys2026/music-crs-baselines
EXPERIMENT_TID = '120-bge-m3-prorank-rerank-devset'
EXPERIMENT_PRED = f'exp/inference/devset/{EXPERIMENT_TID}.json'
import os
if os.path.exists(EXPERIMENT_PRED):
    print(f'Experiment predictions already exist at {EXPERIMENT_PRED}.')
    print('To force re-run: !rm', EXPERIMENT_PRED)
else:
    print('Running experiment inference (~30-45 min)...')
    !python run_inference_devset.py --tid {EXPERIMENT_TID}
%cd /content/recsys2026

In [ ]:
# 8) Score BOTH with the official music-crs-evaluator (this is the metric Blind-A uses).
%cd /content/recsys2026/music-crs-evaluator
for TID in [BASELINE_TID, EXPERIMENT_TID]:
    print(f'--- evaluating {TID} ---')
    !python evaluate_devset.py --tid {TID}
%cd /content/recsys2026

In [ ]:
# 9) Display the nDCG@20 + composite-retrieval comparison + decision.
import json
from pathlib import Path

SCORES_DIR = Path('/content/recsys2026/music-crs-evaluator/exp/scores/devset')
baseline = json.load(open(SCORES_DIR / f'{BASELINE_TID}.json'))
experiment = json.load(open(SCORES_DIR / f'{EXPERIMENT_TID}.json'))

print(f"\n{'Metric':<30} {'baseline (110)':>18} {'BGE-M3 (120)':>18} {'Δ':>14}")
print('-' * 84)
metrics_of_interest = ['ndcg@20', 'ndcg@10', 'ndcg@1', 'catalog_diversity', 'lexical_diversity']
for m in metrics_of_interest:
    b = baseline.get(m)
    e = experiment.get(m)
    if b is None or e is None:
        print(f"{m:<30} {'(missing)':>18} {'(missing)':>18}")
        continue
    delta = e - b
    arrow = '▲' if delta > 0 else ('▼' if delta < 0 else '·')
    print(f"{m:<30} {b:>18.4f} {e:>18.4f} {delta:>+10.4f} {arrow}")

delta_ndcg20 = experiment.get('ndcg@20', 0) - baseline.get('ndcg@20', 0)
print('\n--- decision ---')
if delta_ndcg20 >= 0.02:
    print(f'SHIP IT: Δ nDCG@20 = {delta_ndcg20:+.4f} ≥ +0.02 threshold. Submit to Blind-A.')
elif delta_ndcg20 > 0:
    print(f'KEEP IN ENSEMBLE: Δ nDCG@20 = {delta_ndcg20:+.4f} > 0 but below +0.02. Add to ensemble pool, don\'t burn solo Blind-A slot.')
else:
    print(f'PIVOT: Δ nDCG@20 = {delta_ndcg20:+.4f} ≤ 0. BGE-M3 swap did not help; try Bundle B (Qwen3-4B) or Bundle C (preprocessing).')

# Save the comparison summary for memory/log.
out = {
    'baseline_tid': BASELINE_TID, 'experiment_tid': EXPERIMENT_TID,
    'baseline_scores': baseline, 'experiment_scores': experiment,
    'delta_ndcg@20': delta_ndcg20,
}
Path('/content/recsys2026/data').mkdir(exist_ok=True)
with open('/content/recsys2026/data/phase1_bundle_a_vs_baseline.json', 'w') as f:
    json.dump(out, f, indent=2)
print('\nWrote /content/recsys2026/data/phase1_bundle_a_vs_baseline.json')

In [ ]:
# 10) Optional — commit comparison summary back to the branch.
!git config user.email 'orrimoch@gmail.com'
!git config user.name 'Or Rimoch (Colab)'
!git add data/phase1_bundle_a_vs_baseline.json
!git commit -m 'phase1 bundle a (BGE-M3): scoring comparison vs baseline 110'
# !git push origin {BRANCH}   # uncomment when ready to push